In [32]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
from scipy.interpolate import interp1d

/opt/sanjeev/NOAA/MVT/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [40]:
pws = pd.read_csv('/opt/sanjeev/NOAA/MVT/data/PWS/klch2022-24.csv')
pws = pws[2:]
pws

,obsTimeUtc,imperial.precipRate,imperial.precipTotal
2,1/1/2022 6:14,0.0,0.00
3,1/1/2022 6:19,0.0,0.00
4,1/1/2022 6:24,0.0,0.00
5,1/1/2022 6:29,0.0,0.00
6,1/1/2022 6:34,0.0,0.00
...,...,...,...
295946,10/31/2024 23:39,0.0,1.45
295947,10/31/2024 23:44,0.0,1.45
295948,10/31/2024 23:49,0.0,1.45
295949,10/31/2024 23:54,0.0,1.45


In [81]:
select_cols = ["imperial.precipRate", "imperial.precipTotal"]

irregular_dates = pd.to_datetime(pws["obsTimeUtc"])
data = pws[select_cols].values

print(np.max(pws["imperial.precipRate"].values))
print(np.max(pws["imperial.precipTotal"].values))

print(np.min(pws["imperial.precipRate"].values))
print(np.min(pws["imperial.precipTotal"].values))

7.32
4.17
0.0
0.0


In [82]:
# print(data.shape)
df = pd.DataFrame(data, index=irregular_dates)
# print(df)

# Define the target start time and interval for the regular time series
# resample_start_time = '2022-08-01 00:05'
# resample_start_time = '2022-07-02 00:01' #because radar data starts at 07-02 00:01 
resample_start_time = '2022-01-01 06:14' 
interval = '15T'

# Generate the new regular date range
timestamps_15min = pd.date_range(start=resample_start_time, periods=96359, freq='15T')

shifted_timestamps_15min = timestamps_15min + pd.to_timedelta('2m00s')

# self.ids = self.timestamps = [dt.strftime('%Y%m%d_%H%M') for dt in resample_date_range][:5856]
ids = timestamps = [dt.strftime('%Y%m%d_%H%M') for dt in shifted_timestamps_15min]

print(ids[-1])

# Convert the timestamps to numerical values (e.g., seconds since the epoch) for interpolation
x_original = (df.index - pd.Timestamp("1970-01-01")) // pd.Timedelta('1s')
x_resample = (shifted_timestamps_15min - pd.Timestamp("1970-01-01")) // pd.Timedelta('1s')

# Polynomial interpolation using SciPy for each column
resampled_data = {}
for column in df.columns:
    interp_func = interp1d(x_original, df[column], kind='cubic', fill_value="extrapolate")
    resampled_data[column] = interp_func(x_resample)

# Create a new DataFrame with the resampled data
resampled_df = pd.DataFrame(resampled_data, index=shifted_timestamps_15min)

# Print the resampled DataFrame
# print(resampled_df)
data = resampled_df * 25.4

df = data #5760 for only two months of data
# print(df.keys)

df_np = df.values
df_np[df_np < 0] = 0

df_np = np.around(df_np, 2)
        
print(np.min(df_np))
# print(np.min(df[1].values))    

data = torch.from_numpy(df_np)
        
print(torch.min(data))
print(torch.max(data))
print(data.shape)

20240930_2346
0.0
tensor(0., dtype=torch.float64)
tensor(182.6300, dtype=torch.float64)
torch.Size([96359, 2])


In [61]:
print(resampled_df[0].ne(0).sum())
print(resampled_df[0].eq(0).sum())

print(type(resampled_df))

35394
60965
<class 'pandas.core.frame.DataFrame'>


In [78]:
filtered_df = resampled_df.round(2)
filtered_df = filtered_df[filtered_df[0] > 0]
filtered_df

,0,1
2022-01-01 21:01:00,0.09,0.02
2022-01-01 21:16:00,0.15,0.04
2022-01-02 11:31:00,0.08,0.01
2022-01-05 16:31:00,0.05,0.01
2022-01-06 12:46:00,0.07,0.01
...,...,...
2024-09-22 22:01:00,0.49,0.24
2024-09-22 22:16:00,0.08,0.26
2024-09-25 17:16:00,0.73,0.12
2024-09-25 17:31:00,1.77,0.43


In [80]:
resampled_df.round(2).to_csv('resampled_lkch22-24.csv')